# Camera Calibration and Undistortion

Calibrate a camera using checkerboard images.
    
    The Pinhole Camera Model projects 3D world coordinates (X, Y, Z)
    onto 2D image coordinates (u, v):
        [u, v, 1]^T = K * [R | t] * [X, Y, Z, 1]^T
        
    Where K is the intrinsic camera matrix:
        K = [[fx,  0, cx],
             [ 0, fy, cy],
             [ 0,  0,  1]]
        - fx, fy : focal length in pixel units
        - cx, cy : principal point (optical center)

In [1]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [2]:
calib_folder="calibration_images"
pattern_size=(9, 6)

## 1. Prepare 3D object points in real-world space (assuming Z=0 for flat checkerboard)

In [3]:
cols, rows = pattern_size
objp = np.zeros((cols * rows, 3), np.float32)
objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)

# Arrays to store 3D object points and 2D image points from all checkerboard images
objpoints = []  # 3D points in real world
imgpoints = []  # 2D points in image plane

image_paths = sorted(glob.glob(os.path.join(calib_folder, "*.jpg")))
if not image_paths:
    print(f"[Error] No calibration images found in '{calib_folder}'.")
    
print(f"Found {len(image_paths)} calibration images. Finding chessboard corners...")

valid_images = 0
img_shape = None
sample_img_with_corners = None

for path in image_paths:
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_shape = gray.shape[::-1]  # (width, height)
    
    # Find chessboard corners
    ret, corners = cv2.findChessboardCorners(gray, (cols, rows), None)
    
    if ret:
        objpoints.append(objp)
        
        # Refine corner locations to sub-pixel accuracy
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        corners_subpix = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners_subpix)
        valid_images += 1
        
        # Save the first image with drawn corners for visualization
        if sample_img_with_corners is None:
            sample_img_with_corners = img.copy()
            cv2.drawChessboardCorners(sample_img_with_corners, (cols, rows), corners_subpix, ret)

print(f"Chessboard successfully detected in {valid_images} / {len(image_paths)} images.")

Found 13 calibration images. Finding chessboard corners...
Chessboard successfully detected in 13 / 13 images.


## 2. Run Camera Calibration

In [4]:
ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, img_shape, None, None)

print("\n--- Calibration Results ---")
print("Intrinsic Matrix K:")
print(np.round(camera_matrix, 2))
print(f"Focal Length fx: {camera_matrix[0, 0]:.2f} px, fy: {camera_matrix[1, 1]:.2f} px")
print(f"Optical Center cx: {camera_matrix[0, 2]:.2f} px, cy: {camera_matrix[1, 2]:.2f} px")
print("\nDistortion Coefficients (k1, k2, p1, p2, k3):")
print(np.round(dist_coeffs.ravel(), 5))


--- Calibration Results ---
Intrinsic Matrix K:
[[536.07   0.   342.37]
 [  0.   536.02 235.54]
 [  0.     0.     1.  ]]
Focal Length fx: 536.07 px, fy: 536.02 px
Optical Center cx: 342.37 px, cy: 235.54 px

Distortion Coefficients (k1, k2, p1, p2, k3):
[-0.26509 -0.04674  0.00183 -0.00031  0.25231]


## 3. Calculate Mean Reprojection Error (Quantitative Metric)

In [5]:
total_error = 0
for i in range(len(objpoints)):
    projected_points, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], camera_matrix, dist_coeffs)
    pts_true = imgpoints[i].reshape(-1, 2)
    pts_proj = projected_points.reshape(-1, 2)
    # Euclidean distance between true and projected 2D points
    error = np.mean(np.linalg.norm(pts_true - pts_proj, axis=1))
    total_error += error
mean_error = total_error / len(objpoints)
print(f"\nMean Re-projection Error: {mean_error:.4f} pixels")


Mean Re-projection Error: 0.2346 pixels


## 4. Undistort a sample image

In [6]:
test_img = cv2.imread(image_paths[0])
undistorted_img = cv2.undistort(test_img, camera_matrix, dist_coeffs)

## 5. Plot and save comparison

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(sample_img_with_corners, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Detected Corners ({cols}x{rows})")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Original (Distorted)")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB))
axes[2].set_title("Calibrated (Undistorted)")
axes[2].axis("off")

plt.tight_layout()
out_path = os.path.join("outputs", "calibration_undistortion.png")
plt.savefig(out_path, dpi=150)
plt.close()
print(f"[Saved Visualization]: {out_path}")


[Saved Visualization]: outputs\calibration_undistortion.png
